# Lab 2: Policy Gradient Methods

## TDDE78 — Deep Reinforcement Learning
### Linköping University, Spring 2026

---

In this lab you will implement two foundational policy gradient algorithms:
- **REINFORCE** (Williams, 1992) — episodic Monte Carlo policy gradient with optional value baseline
- **PPO** (Schulman et al., 2017) — proximal policy optimization with clipped surrogate objective and GAE

The lab is divided into:
- **Part A** — Implementation (fill in the TODO sections)
- **Part B** — Experiments (run systematic evaluations and analyze results)

**Instructions:** Complete all cells marked with `# TODO`. Do not modify the provided helper code unless explicitly asked.

## Setup

Run the cell below to import all necessary libraries and verify your environment.

In [ ]:
import os
import sys
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import Video, display
import warnings
warnings.filterwarnings('ignore')

# Import lab utilities
from networks import DiscreteActorCritic
from utils import compute_returns, compute_gae, RolloutBuffer

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Resolve experiments directory relative to this notebook
_here = globals().get('__vsc_ipynb_file__', os.path.abspath(''))
_nb_dir = os.path.dirname(_here) if os.path.isfile(_here) else _here
EXPERIMENTS_DIR = os.path.normpath(os.path.join(_nb_dir, '..', 'experiments'))
print(f'Experiments directory: {EXPERIMENTS_DIR}')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(42)
print('Setup complete!')

---

# Part A — Implementation

---

## A.1 — Explore the Environment

Before implementing the algorithms, let's understand the environment we will work with.

**CartPole-v1** — balance a pole on a cart by pushing left or right (discrete actions).

In [ ]:
env_cp = gym.make('CartPole-v1')
print('=== CartPole-v1 ===')
print(f'  Observation space: {env_cp.observation_space}  shape={env_cp.observation_space.shape}')
print(f'  Action space:      {env_cp.action_space}')
obs, _ = env_cp.reset(seed=42)
print(f'  Sample obs: {obs}')
env_cp.close()

## A.2 — Actor-Critic Network

Open `networks.py` and implement `DiscreteActorCritic`.

It uses a **shared backbone** (2-layer MLP with Tanh) with separate **policy** and **value** heads:

```
state → Linear(64) → Tanh → Linear(64) → Tanh → [features]
                                                    ├── policy_head → logits (Categorical)
                                                    └── value_head  → V(s)
```

After implementing, run the test below.

In [ ]:
# Test DiscreteActorCritic
from networks import DiscreteActorCritic

net_d = DiscreteActorCritic(state_dim=4, action_dim=2).to(device)
test_state = torch.randn(8, 4).to(device)
action, log_prob, entropy, value = net_d.get_action(test_state)

assert action.shape == (8,),    f'action shape should be (8,), got {action.shape}'
assert log_prob.shape == (8,),  f'log_prob shape should be (8,), got {log_prob.shape}'
assert entropy.shape == (8,),   f'entropy shape should be (8,), got {entropy.shape}'
assert value.shape == (8, 1),   f'value shape should be (8, 1), got {value.shape}'

# Test that providing an action gives the correct log_prob
fixed_action = torch.zeros(8, dtype=torch.long).to(device)
_, lp_fixed, _, _ = net_d.get_action(test_state, fixed_action)
assert lp_fixed.shape == (8,)

n_params = sum(p.numel() for p in net_d.parameters())
print(f'DiscreteActorCritic: action={action.shape}, log_prob={log_prob.shape}, '
      f'entropy={entropy.shape}, value={value.shape}, params={n_params:,}')
print('DiscreteActorCritic test passed!')

## A.3 — Utility Functions

Open `utils.py` and implement `compute_returns`, `compute_gae`, and `RolloutBuffer`.

Run the tests below to verify each component.

In [ ]:
# Test compute_returns
from utils import compute_returns

# Simple episode: [1, 1, 1] with gamma=0.99
# G_0 = 1 + 0.99 + 0.99^2 = 2.9701
# G_1 = 1 + 0.99           = 1.99
# G_2 = 1
test_rewards = [1.0, 1.0, 1.0]
returns = compute_returns(test_rewards, gamma=0.99)

assert returns.shape == (3,), f'Expected shape (3,), got {returns.shape}'
assert returns.dtype == np.float32, f'Expected float32, got {returns.dtype}'
assert abs(returns[2] - 1.0) < 1e-5, f'G_2 should be 1.0, got {returns[2]}'
assert abs(returns[1] - 1.99) < 1e-4, f'G_1 should be 1.99, got {returns[1]}'
assert abs(returns[0] - 2.9701) < 1e-4, f'G_0 should be 2.9701, got {returns[0]}'
print(f'compute_returns: {returns}  (expected [2.970, 1.990, 1.000])')
print('compute_returns test passed!')

In [ ]:
# Test compute_gae
from utils import compute_gae

rewards = [1.0, 1.0, 1.0]
values  = [0.5, 0.5, 0.5]
dones   = [False, False, True]  # episode ends at last step
advantages, returns = compute_gae(rewards, values, dones,
                                   last_value=0.0, gamma=0.99, gae_lambda=0.95)

assert advantages.shape == (3,), f'Expected shape (3,), got {advantages.shape}'
assert returns.shape == (3,),    f'Expected shape (3,), got {returns.shape}'
assert advantages.dtype == np.float32
# Returns should be close to advantages + values
np.testing.assert_allclose(returns, advantages + np.array(values, dtype=np.float32), rtol=1e-5)
print(f'advantages: {advantages}')
print(f'returns:    {returns}')
print('compute_gae test passed!')

In [ ]:
# Test RolloutBuffer
from utils import RolloutBuffer

buf = RolloutBuffer(n_steps=100, state_dim=4, action_dim=2, is_continuous=False)

for i in range(60):
    state   = np.random.randn(4).astype(np.float32)
    action  = np.random.randint(0, 2)
    reward  = float(np.random.randn())
    done    = (i % 20 == 19)
    log_prob = float(np.random.randn())
    value   = float(np.random.randn())
    buf.store(state, action, reward, done, log_prob, value)

assert len(buf) == 60, f'Expected 60, got {len(buf)}'

buf.compute_returns_and_advantages(last_value=0.0, gamma=0.99, gae_lambda=0.95)
assert buf.advantages is not None, 'advantages not set'
assert buf.returns is not None, 'returns not set'

batches = list(buf.get_batches(batch_size=16))
assert len(batches) >= 3, f'Expected >=3 batches, got {len(batches)}'
states_b, actions_b, log_probs_b, advs_b, rets_b = batches[0]
assert states_b.shape[1] == 4, f'State dim should be 4, got {states_b.shape[1]}'
assert actions_b.dtype == torch.int64, f'Discrete actions should be int64'
print('RolloutBuffer test passed!')

## A.4 — REINFORCE Agent

Implement the REINFORCE agent below.

The agent collects **complete episodes**, computes discounted returns $G_t$, and updates the policy with:

$$\mathcal{L}(\theta) = -\frac{1}{T} \sum_t \log \pi_\theta(a_t \mid s_t) \cdot (G_t - V(s_t)) + c_V \cdot \mathcal{L}_V$$

When `use_baseline=False`, the advantage is simply $G_t$ (no subtraction).

In [ ]:
class REINFORCEAgent:
    """
    REINFORCE agent with optional learned baseline.

    Args:
        state_dim    (int):   Dimension of state space.
        action_dim   (int):   Number of discrete actions.
        lr           (float): Learning rate.
        gamma        (float): Discount factor.
        use_baseline (bool):  If True, subtract V(s_t) from returns.
        value_coef   (float): Coefficient for value loss.
        seed         (int):   Random seed.
    """

    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 use_baseline=True, value_coef=0.5, seed=42):
        self.gamma = gamma
        self.use_baseline = use_baseline
        self.value_coef = value_coef

        # =====================================================================
        # TODO: Initialize:
        #   1. self.network — DiscreteActorCritic(state_dim, action_dim) on device
        #   2. self.optimizer — Adam with the given lr
        # =====================================================================
        raise NotImplementedError("Initialize REINFORCEAgent")

    def select_action(self, state):
        """Sample an action stochastically (no gradient)."""
        # =====================================================================
        # TODO: Convert state to tensor, call self.network.get_action() with
        #       torch.no_grad(), and return action.item(), log_prob.item().
        # =====================================================================
        raise NotImplementedError("Implement REINFORCEAgent.select_action()")

    def update(self, states, actions, rewards):
        """
        Perform one gradient update on a complete episode.

        Returns:
            pg_loss (float), value_loss (float)
        """
        # =====================================================================
        # TODO: Implement the REINFORCE update.
        #
        # Step 1: Compute raw returns and a normalized copy for value training
        #   returns   = compute_returns(rewards, self.gamma)
        #   returns_t = torch.FloatTensor(returns).to(device)
        #   returns_norm = (returns_t - returns_t.mean()) / (returns_t.std() + 1e-8)
        #                  (only if len > 1, otherwise returns_norm = returns_t)
        #
        # Step 2: Re-evaluate log_probs and values WITH gradients
        #   states_t  = torch.FloatTensor(np.array(states)).to(device)
        #   actions_t = torch.LongTensor(actions).to(device)
        #   _, log_probs, _, values = self.network.get_action(states_t, actions_t)
        #   values = values.squeeze(-1)
        #
        # Step 3: Compute advantages
        #   If use_baseline:
        #     advantages = returns_t - values.detach()   # raw returns minus baseline
        #     if len > 1:
        #         advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        #     value_loss = F.mse_loss(values, returns_norm)
        #   Else:
        #     advantages = returns_norm                  # normalized returns as signal
        #     value_loss = torch.tensor(0.0, device=device)
        #
        # Key: normalize returns for value training, advantages for policy gradient.
        # Avoid normalizing returns and then subtracting a non-normalized baseline
        # (that double-normalizes and weakens the gradient signal).
        #
        # Step 4: pg_loss = -(log_probs * advantages).mean()
        # Step 5: loss = pg_loss + value_coef * value_loss
        # Step 6: zero_grad / backward / clip_grad_norm_(0.5) / step
        # Return pg_loss.item(), value_loss.item()
        # =====================================================================
        raise NotImplementedError("Implement REINFORCEAgent.update()")

print("REINFORCEAgent class defined!")

## A.5 — REINFORCE Training Loop

Implement the full training loop for REINFORCE. The loop collects one **complete episode** per iteration and calls `agent.update()` at the end.

In [ ]:
def train_reinforce(env_name='CartPole-v1', num_episodes=600, lr=1e-3,
                    gamma=0.99, use_baseline=True, value_coef=0.5,
                    seed=42, solve_threshold=None):
    """
    Train a REINFORCE agent.

    Args:
        env_name        (str):   Gymnasium environment name.
        num_episodes    (int):   Maximum number of training episodes.
        lr              (float): Learning rate.
        gamma           (float): Discount factor.
        use_baseline    (bool):  Whether to use the value baseline.
        value_coef      (float): Coefficient for value loss.
        seed            (int):   Random seed.
        solve_threshold (float): Stop early when avg reward (last 100 eps) >= this.

    Returns:
        dict with keys: episode_rewards, pg_losses, value_losses, agent
    """
    set_seed(seed)
    env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = REINFORCEAgent(state_dim, action_dim, lr=lr, gamma=gamma,
                           use_baseline=use_baseline, value_coef=value_coef, seed=seed)

    episode_rewards, pg_losses, value_losses = [], [], []

    # =========================================================================
    # TODO: Implement the REINFORCE training loop.
    #
    # For each episode:
    #   1. Reset the environment: obs, _ = env.reset(seed=seed + episode)
    #   2. Initialize states=[], actions=[], rewards=[], episode_reward=0.0
    #   3. Step through the episode (up to 10000 steps):
    #      a. Select action: action, _ = agent.select_action(obs)
    #      b. Step: next_obs, reward, terminated, truncated, _ = env.step(action)
    #      c. Append to states, actions, rewards; episode_reward += reward
    #      d. obs = next_obs
    #      e. if terminated or truncated: break
    #   4. Update: pg_loss, val_loss = agent.update(states, actions, rewards)
    #   5. Append to episode_rewards, pg_losses, value_losses
    #   6. Every 50 episodes, print:
    #      f"Episode {episode+1}/{num_episodes} | Avg Reward (50): {avg:.1f} | PG Loss: {pg_loss:.4f}"
    #   7. Early stopping if avg reward (last 100 eps) >= solve_threshold
    # =========================================================================
    raise NotImplementedError("Implement train_reinforce training loop")

    env.close()
    return {'episode_rewards': episode_rewards, 'pg_losses': pg_losses,
            'value_losses': value_losses, 'agent': agent}

print("train_reinforce defined!")

## A.6 — Plotting Utilities

Helper functions for visualizing training results. No changes needed here.

In [ ]:
def _save_plot(fig, title):
    try:
        plots_dir = os.path.join(EXPERIMENTS_DIR, 'plots')
        os.makedirs(plots_dir, exist_ok=True)
        fname = title.lower().replace(' ', '_').replace('/', '_').replace('—', '').replace(':', '').strip('_') + '.png'
        fig.savefig(os.path.join(plots_dir, fname), dpi=150, bbox_inches='tight')
        print(f'Plot saved: {os.path.join(plots_dir, fname)}')
    except Exception as e:
        print(f'Could not save plot: {e}')


def plot_reinforce_results(results, title='REINFORCE Training', window=50):
    """Plot REINFORCE training curves (episode-based)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    rewards = results['episode_rewards']

    axes[0].plot(rewards, alpha=0.3, color='blue', label='Raw')
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode='valid')
        axes[0].plot(range(window - 1, len(rewards)), ma, color='blue', label=f'{window}-ep avg')
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
    axes[0].set_title(f'{title} — Episode Rewards'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(results['pg_losses'], alpha=0.5, color='red')
    axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Policy Loss')
    axes[1].set_title(f'{title} — Policy Gradient Loss'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(results['value_losses'], alpha=0.5, color='green')
    axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Value Loss')
    axes[2].set_title(f'{title} — Value Loss'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    _save_plot(fig, title)
    plt.show()


def plot_ppo_results(results, title='PPO Training', window=20):
    """Plot PPO training curves (update-based)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    rewards = results['update_rewards']

    axes[0].plot(rewards, alpha=0.4, color='blue', label='Eval reward')
    if len(rewards) >= window:
        ma = np.convolve(rewards, np.ones(window) / window, mode='valid')
        axes[0].plot(range(window - 1, len(rewards)), ma, color='blue', label=f'{window}-update avg')
    axes[0].set_xlabel('PPO Update'); axes[0].set_ylabel('Reward')
    axes[0].set_title(f'{title} — Eval Reward'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(results['policy_losses'], alpha=0.5, color='red')
    axes[1].set_xlabel('PPO Update'); axes[1].set_ylabel('Policy Loss')
    axes[1].set_title(f'{title} — Policy Loss'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(results['entropies'], alpha=0.5, color='purple')
    axes[2].set_xlabel('PPO Update'); axes[2].set_ylabel('Entropy')
    axes[2].set_title(f'{title} — Policy Entropy'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    _save_plot(fig, title)
    plt.show()


def plot_comparison(all_results, title='Comparison', window=50, x_label='Episode', reward_key='episode_rewards'):
    """Plot reward comparison across methods/seeds (mean ± std)."""
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['blue', 'red', 'green', 'orange', 'purple']

    for idx, (name, results_list) in enumerate(all_results.items()):
        color = colors[idx % len(colors)]
        min_len = min(len(r[reward_key]) for r in results_list)
        all_rewards = np.array([r[reward_key][:min_len] for r in results_list])

        if min_len >= window:
            smoothed = np.array([np.convolve(row, np.ones(window) / window, mode='valid')
                                 for row in all_rewards])
            mean, std = smoothed.mean(axis=0), smoothed.std(axis=0)
            x = range(window - 1, min_len)
        else:
            mean, std = all_rewards.mean(axis=0), all_rewards.std(axis=0)
            x = range(min_len)

        ax.plot(x, mean, color=color, label=name)
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.15)

    ax.set_xlabel(x_label); ax.set_ylabel('Reward'); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    _save_plot(fig, title)
    plt.show()

print('Plotting utilities defined!')

## A.7 — Train REINFORCE on CartPole-v1

Verify your REINFORCE implementation. The agent should reach average reward ≥ 475 within ~600 episodes.

In [ ]:
set_seed(42)
results_rf = train_reinforce(
    env_name='CartPole-v1',
    num_episodes=2000,
    lr=1e-3,
    use_baseline=True,
    solve_threshold=475,
    seed=42,
)

plot_reinforce_results(results_rf, title='REINFORCE on CartPole-v1')

avg = np.mean(results_rf['episode_rewards'][-100:])
print(f'\nAverage reward (last 100 eps): {avg:.1f}')
print('CartPole SOLVED!' if avg >= 475 else 'Not yet solved — check your implementation.')

## Visualise — REINFORCE Agent on CartPole-v1

In [ ]:
def record_video(agent, env_name, video_dir, episode_trigger=lambda ep: ep == 0, seed=42):
    """Record one episode of the agent and return the video path."""
    os.makedirs(video_dir, exist_ok=True)
    env = gym.make(env_name, render_mode='rgb_array')
    env = gym.wrappers.RecordVideo(env, video_folder=video_dir,
                                   episode_trigger=episode_trigger,
                                   disable_logger=True)
    obs, _ = env.reset(seed=seed)
    ep_reward = 0.0
    for _ in range(10000):
        state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        with torch.no_grad():
            action, _, _, _ = agent.network.get_action(state_t)
        action_np = action.cpu().numpy().squeeze(0)
        obs, reward, terminated, truncated, _ = env.step(int(action_np))
        ep_reward += reward
        if terminated or truncated:
            break
    env.close()
    print(f'Episode reward: {ep_reward:.1f}')
    videos = sorted([f for f in os.listdir(video_dir) if f.endswith('.mp4')])
    return os.path.join(video_dir, videos[-1]) if videos else None


video_dir = os.path.join(EXPERIMENTS_DIR, 'videos', 'reinforce_cartpole')
video_path = record_video(results_rf['agent'], 'CartPole-v1', video_dir, seed=42)

if video_path:
    display(Video(video_path, embed=True, width=400))
else:
    print('No video found — check that ffmpeg is installed (sudo apt install ffmpeg).')

## A.8 — PPO Agent

Implement the PPO agent. This is the core of the lab.

PPO operates in a **collect → update** cycle:
1. **collect_rollout**: gather `n_steps` transitions using the current policy, then compute GAE
2. **update**: run `n_epochs` passes over the rollout data with the clipped PPO objective

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) A_t, \; \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) A_t \right) \right]$$

In [ ]:
class PPOAgent:
    """
    PPO agent for discrete action spaces (CartPole-v1).

    Args:
        state_dim     (int):   Dimension of state space.
        action_dim    (int):   Number of discrete actions.
        lr            (float): Learning rate.
        gamma         (float): Discount factor.
        gae_lambda    (float): GAE lambda (0=TD, 1=MC).
        clip_eps      (float): PPO clipping parameter epsilon.
        n_epochs      (int):   Update epochs per rollout.
        batch_size    (int):   Mini-batch size.
        value_coef    (float): Weight for value loss.
        entropy_coef  (float): Weight for entropy bonus.
        n_steps       (int):   Environment steps per rollout.
        max_grad_norm (float): Gradient clipping threshold.
        seed          (int):   Random seed.
    """

    def __init__(self, state_dim, action_dim,
                 lr=2.5e-4, gamma=0.99, gae_lambda=0.95, clip_eps=0.2,
                 n_epochs=4, batch_size=64, value_coef=0.5, entropy_coef=0.01,
                 n_steps=512, max_grad_norm=0.5, seed=42):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_eps = clip_eps
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.n_steps = n_steps
        self.max_grad_norm = max_grad_norm

        # =====================================================================
        # TODO: Initialize the following:
        #   1. self.buffer — RolloutBuffer(n_steps, state_dim, action_dim, is_continuous=False)
        #   2. self.network — DiscreteActorCritic(state_dim, action_dim), move to device
        #   3. self.optimizer — Adam(lr=lr, eps=1e-5)
        #   4. self.obs = None  (env state carried between rollouts, set by train_ppo)
        # =====================================================================
        raise NotImplementedError("Initialize PPOAgent")

    def select_action(self, state):
        """
        Sample an action from the current policy (no gradient).

        Returns:
            action_np (np.ndarray): Sampled action as numpy scalar.
            log_prob  (float):      Log probability of the action.
            value     (float):      State value estimate V(s).
        """
        # =====================================================================
        # TODO: Convert state to tensor (unsqueeze batch dim), call
        #       self.network.get_action() inside torch.no_grad(), then:
        #         action_np = action.cpu().numpy().squeeze(0)
        #       Return action_np, log_prob.item(), value.item()
        # =====================================================================
        raise NotImplementedError("Implement PPOAgent.select_action()")

    def collect_rollout(self, env):
        """
        Collect n_steps transitions and compute GAE advantages.

        Uses self.obs to carry env state between rollouts (CleanRL-style),
        so the agent continues from where the previous rollout left off.

        Args:
            env: A Gymnasium environment instance.

        Returns:
            float: Mean episode reward over completed episodes in this rollout.
        """
        # =====================================================================
        # TODO: Collect n_steps transitions using self.obs (not a local obs).
        #
        # 1. self.buffer.clear()
        # 2. ep_rewards = []; current_ep_reward = 0.0
        # 3. For _ in range(self.n_steps):
        #      a. action_np, log_prob, value = self.select_action(self.obs)
        #      b. next_obs, reward, terminated, truncated, _ = env.step(int(action_np))
        #         done = terminated or truncated
        #      c. self.buffer.store(self.obs, action_np, reward, done, log_prob, value)
        #      d. current_ep_reward += reward; self.obs = next_obs
        #      e. if done:
        #           ep_rewards.append(current_ep_reward)
        #           current_ep_reward = 0.0
        #           self.obs, _ = env.reset()
        # 4. Bootstrap last value using self.obs (the current obs after the loop):
        #      state_t = torch.FloatTensor(self.obs).unsqueeze(0).to(device)
        #      with torch.no_grad():
        #          _, _, _, last_value = self.network.get_action(state_t)
        # 5. self.buffer.compute_returns_and_advantages(
        #        last_value.item(), self.gamma, self.gae_lambda)
        # 6. Return np.mean(ep_rewards) if ep_rewards else 0.0
        # =====================================================================
        raise NotImplementedError("Implement PPOAgent.collect_rollout()")

    def update(self):
        """
        Run n_epochs of PPO updates over the collected rollout.

        Returns:
            Tuple of (mean_policy_loss, mean_value_loss, mean_entropy)
        """
        # =====================================================================
        # TODO: Implement the PPO update loop.
        #
        # For _ in range(self.n_epochs):
        #   For each batch from self.buffer.get_batches(self.batch_size):
        #     Unpack: states, actions, old_log_probs, advantages, returns, old_values = batch
        #     Move all tensors to device.
        #
        #     1. Normalize advantages per mini-batch (CleanRL standard):
        #          advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        #
        #     2. Re-evaluate: _, new_log_probs, entropy, new_values =
        #                         self.network.get_action(states, actions)
        #        new_values = new_values.squeeze(-1)
        #
        #     3. PPO clipped objective:
        #          log_ratio = new_log_probs - old_log_probs
        #          ratio     = log_ratio.exp()
        #          surr1 = ratio * advantages
        #          surr2 = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages
        #          policy_loss = -torch.min(surr1, surr2).mean()
        #
        #     4. Value loss with clipping (prevents V from drifting per epoch):
        #          v_loss_unclip = F.mse_loss(new_values, returns)
        #          v_clipped = old_values + torch.clamp(
        #              new_values - old_values, -self.clip_eps, self.clip_eps)
        #          v_loss_clipped = F.mse_loss(v_clipped, returns)
        #          value_loss = torch.max(v_loss_unclip, v_loss_clipped)
        #
        #     5. Entropy bonus: entropy_loss = -entropy.mean()
        #
        #     6. Total loss:
        #          loss = policy_loss + value_coef * value_loss + entropy_coef * entropy_loss
        #
        #     7. optimizer.zero_grad() / loss.backward()
        #        nn.utils.clip_grad_norm_(self.network.parameters(), max_grad_norm)
        #        optimizer.step()
        #
        # Return (mean_policy_loss, mean_value_loss, mean_entropy)
        # =====================================================================
        raise NotImplementedError("Implement PPOAgent.update()")

print("PPOAgent class defined!")

## A.9 — PPO Training Loop

Implement `train_ppo`. Unlike REINFORCE, PPO uses a fixed number of **rollout-update cycles** rather than episodes, so tracking is update-based.

In [ ]:
def evaluate_ppo(agent, env, num_episodes=5, seed=0):
    """Evaluate a PPO agent (discrete actions)."""
    rewards = []
    for ep in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_reward = 0.0
        for _ in range(10000):
            state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            with torch.no_grad():
                action, _, _, _ = agent.network.get_action(state_t)
            obs, reward, terminated, truncated, _ = env.step(int(action.item()))
            ep_reward += reward
            if terminated or truncated:
                break
        rewards.append(ep_reward)
    return np.mean(rewards)


def train_ppo(env_name='CartPole-v1', num_updates=200, n_steps=512,
              seed=42, solve_threshold=None, anneal_lr=True, **agent_kwargs):
    """
    Train a PPO agent with optional linear LR annealing (CleanRL-style).

    Args:
        env_name        (str):  Gymnasium environment name.
        num_updates     (int):  Number of rollout-update cycles.
        n_steps         (int):  Environment steps per rollout.
        seed            (int):  Random seed.
        solve_threshold (float): Stop early when avg eval reward >= this.
        anneal_lr       (bool): Linearly decay LR from initial value to 0.
        **agent_kwargs:         Passed to PPOAgent (lr, gae_lambda, clip_eps, etc.)

    Returns:
        dict with keys: update_rewards, policy_losses, value_losses, entropies, agent
    """
    set_seed(seed)
    env = gym.make(env_name)
    eval_env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = PPOAgent(state_dim, action_dim, n_steps=n_steps, seed=seed, **agent_kwargs)

    # Initialize env state — carried between rollouts (CleanRL-style)
    agent.obs, _ = env.reset(seed=seed)

    # Linear LR annealing: lr * (1 - update / num_updates) — same as CleanRL
    if anneal_lr:
        scheduler = torch.optim.lr_scheduler.LambdaLR(
            agent.optimizer,
            lr_lambda=lambda step: 1.0 - step / num_updates
        )

    update_rewards, policy_losses, value_losses, entropies = [], [], [], []

    # =========================================================================
    # TODO: Implement the PPO training loop.
    #
    # For update in range(num_updates):
    #   1. agent.collect_rollout(env)   ← uses agent.obs internally
    #   2. pg_loss, val_loss, entropy = agent.update()
    #   3. if anneal_lr: scheduler.step()
    #   4. eval_reward = evaluate_ppo(agent, eval_env, num_episodes=5, seed=seed)
    #   5. Append to tracking lists
    #   6. Every 20 updates, print:
    #      current_lr = agent.optimizer.param_groups[0]['lr']
    #      f"Update {update+1}/{num_updates} | Avg Eval Reward (20): {avg:.1f} | LR: {current_lr:.2e}"
    #   7. Early stopping if avg eval reward (last 10 updates) >= solve_threshold
    # =========================================================================
    raise NotImplementedError("Implement train_ppo training loop")

    env.close()
    eval_env.close()
    return {'update_rewards': update_rewards, 'policy_losses': policy_losses,
            'value_losses': value_losses, 'entropies': entropies, 'agent': agent}

print("train_ppo and evaluate_ppo defined!")

## A.10 — Train PPO on CartPole-v1

Verify your PPO implementation on CartPole.

In [ ]:
set_seed(42)
results_ppo_cp = train_ppo(
    env_name='CartPole-v1',
    num_updates=1000,
    n_steps=2048,
    solve_threshold=475,
    seed=42,
    lr=3e-4,
    gae_lambda=0.95,
    clip_eps=0.2,
    n_epochs=4,
    batch_size=64,
    entropy_coef=0.01,
)

plot_ppo_results(results_ppo_cp, title='PPO on CartPole-v1')

avg = np.mean(results_ppo_cp['update_rewards'][-10:])
solved = avg >= 475 or len(results_ppo_cp['update_rewards']) < 1000
print(f'\nAverage eval reward (last 10 updates): {avg:.1f}')
print('CartPole SOLVED!' if solved else 'Not yet solved — check your implementation.')

## Visualise — PPO Agent on CartPole-v1

In [ ]:
video_dir = os.path.join(EXPERIMENTS_DIR, 'videos', 'ppo_cartpole')
video_path = record_video(results_ppo_cp['agent'], 'CartPole-v1', video_dir, seed=42)

if video_path:
    display(Video(video_path, embed=True, width=400))
else:
    print('No video found — check that ffmpeg is installed (sudo apt install ffmpeg).')

---

# Part B — Experiments

---

Now that your implementation works, run systematic experiments to understand the algorithms.

**Important:** For all experiments, run at least **3 different seeds** and report mean ± std.

## B.1 — REINFORCE vs PPO on CartPole-v1

Compare the two algorithms with seeds [42, 123, 456].

In [ ]:
# =====================================================================
# TODO: Train REINFORCE (with baseline, num_episodes=2000) and PPO
#       (num_updates=600, n_steps=2048, n_epochs=10, lr=3e-4) on
#       CartPole-v1 with seeds [42, 123, 456].
#       Report and compare final performance.
#
# Note: REINFORCE and PPO use different x-axes (episodes vs updates),
# so plot them separately using plot_comparison with the correct
# reward_key ('episode_rewards' or 'update_rewards').
# =====================================================================
raise NotImplementedError("Run REINFORCE vs PPO comparison")

## B.2 — Ablation: Baseline in REINFORCE

Compare REINFORCE with and without the value baseline. Does it reduce variance?

In [ ]:
# =====================================================================
# TODO: Run REINFORCE with use_baseline=True and use_baseline=False
#       on CartPole-v1 (num_episodes=2000) with seeds [42, 123, 456].
#       Plot comparison and analyze the variance difference.
# =====================================================================
raise NotImplementedError("Run baseline ablation")

## B.3 — Ablation: GAE Lambda

GAE interpolates between one-step TD ($\lambda = 0$, low variance, high bias) and Monte Carlo ($\lambda = 1$, high variance, zero bias).

Test lambda values: **0.0**, **0.5**, **0.9**, **0.95**, **1.0**

In [ ]:
# =====================================================================
# TODO: Run PPO on CartPole-v1 (num_updates=600, n_steps=2048,
#       n_epochs=4, lr=3e-4, NO solve_threshold) with 3 key lambda
#       values and seeds [42, 123, 456].
#
#   gae_lambdas = [0.0, 0.95, 1.0]
#     - 0.0  → pure one-step TD (low variance, high bias)
#     - 0.95 → default PPO setting
#     - 1.0  → Monte Carlo (high variance, no bias)
#
# Use n_epochs=4 (standard). n_epochs=10 causes V to shift too much
# per update, making multi-step lambda=0.95 advantages stale and
# artificially favouring lambda=0.
#
# Do NOT pass solve_threshold — you want to see the full 600-update
# learning curve for each lambda, not cut it short at the first spike.
#
# Plot with plot_comparison(..., reward_key='update_rewards') and
# report mean ± std for each lambda. Which value works best and why?
# =====================================================================
raise NotImplementedError("Run GAE lambda ablation")

---

## Summary

**TODO:** Write a brief summary of your findings (double-click to edit).

1. **B.1 — REINFORCE vs PPO:** How do the two algorithms compare in terms of convergence speed and variance? What structural property of PPO explains the difference?
2. **B.2 — Baseline ablation:** How much does the value baseline help REINFORCE? Describe the difference in learning curves (with and without baseline).
3. **B.3 — GAE lambda:** Which lambda value performed best? Describe the bias-variance trade-off you observed — what happens at the extremes (λ = 0 and λ = 1)?

---

**Lab designed by Amath Sow:** amath.sow@liu.se

**TDDE78 — Deep Reinforcement Learning, Linköping University, Spring 2026**